# Putinformatie opvragen

In [ ]:
import dawacotools as dt

mpcode = "MOCK001"  # echte DAWACO-data: vervang door je eigen putcode
mpcode_2 = "MOCK002"  # echte DAWACO-data: vervang door je tweede eigen putcode
partial_mpcode = "MOCK00"  # echte DAWACO-data: vervang door je eigen putcode-prefix
flow_mpcode_1 = "MOCK001"  # echte DAWACO-data: vervang door je eerste eigen putcode
flow_mpcode_2 = "MOCK002"  # echte DAWACO-data: vervang door je tweede eigen putcode

# Voor lokaal testen met echte data: vervang de MOCK-waarden hierboven door eigen DAWACO-putcodes.

Probeer de volgende manieren om de functie aan te roepen en geef antwoord op de vragen:

Hoeveel putten staan er in de database?

```mps = dt.get_daw_mps()```

Hoeveel putten staan er in de database met putcode `mpcode`?

```mps = dt.get_daw_mps(mpcode=mpcode)  # lokaal: gebruik je eigen putcode```

Hoeveel putten staan er in de database waarvan de putcode begint met `partial_mpcode`?

```mps = dt.get_daw_mps(mpcode=partial_mpcode, partial_match_mpcode=True)  # lokaal: gebruik je eigen putcode-prefix```

Vraag de informatie van `mpcode` en `mpcode_2` tegelijkertijd op

```mps = dt.get_daw_mps(mpcode=[mpcode, mpcode_2])  # lokaal: gebruik eigen putcodes```

In [ ]:
mps = dt.get_daw_mps()
display(mps)

mps = dt.get_daw_mps(mpcode=mpcode, partial_match_mpcode=False)
display(mps)

mps = dt.get_daw_mps(mpcode=partial_mpcode, partial_match_mpcode=True)
display(mps)

mps = dt.get_daw_mps(mpcode=[mpcode, mpcode_2], partial_match_mpcode=False)
display(mps)

# Filterinformatie opvragen

In [ ]:
# Filterinformatie vraag je op met dt.get_daw_filters.

Haal informatie op van filter 1 van `mpcode`. 0 zou pompput moeten zijn en 1 het eerste observatiefilter.

```filters = dt.get_daw_filters(mpcode=mpcode, filternr=1)  # lokaal: gebruik je eigen putcode```

Haal informatie op van filter 1 en 2 van `mpcode`.

```filters = dt.get_daw_filters(mpcode=mpcode, filternr=[1.0, 2.0])  # lokaal: gebruik je eigen putcode```

Haal informatie op van alle filters van `mpcode`.

```filters = dt.get_daw_filters(mpcode=mpcode)  # lokaal: gebruik je eigen putcode```

In [ ]:
filters = dt.get_daw_filters(mpcode=mpcode, filternr=1, partial_match_mpcode=False)
display(filters)

filters = dt.get_daw_filters(mpcode=mpcode, filternr=[1.0, 2.0], partial_match_mpcode=False)
display(filters)

filters = dt.get_daw_filters(mpcode=mpcode, partial_match_mpcode=False)
display(filters)

# Grondwaterstanden opvragen

- Haal grondwaterstanden op van `flow_mpcode_1` en `flow_mpcode_2` (lokaal met echte DAWACO-data: vervang beide variabelen door eigen putcodes)
- Plot beiden in 1 plaatje

In [ ]:
import matplotlib.pyplot as plt

# Bereken het verhang van de grondwaterstand tussen `flow_mpcode_1` en `flow_mpcode_2` (dh / dx)

Tip: Coordinaten staan in de putinformatie als Xcoord en Ycoord

In [ ]:
gws_1 = dt.get_daw_ts_stijghgt(mpcode=flow_mpcode_1, filternr=1)
gws_2 = dt.get_daw_ts_stijghgt(mpcode=flow_mpcode_2, filternr=1)

ax = gws_1.plot(label=f"{flow_mpcode_1}F1")
gws_2.plot(ax=ax, label=f"{flow_mpcode_2}F1")
ax.set_ylabel("m NAP")
ax.legend()
plt.close(ax.figure)

filters = dt.get_daw_filters(
    mpcode=[flow_mpcode_1, flow_mpcode_2],
    filternr=1,
    partial_match_mpcode=False,
)
delta_h = gws_2 - gws_1
dx = filters.geometry.iloc[0].distance(filters.geometry.iloc[1])
gradient = delta_h / dx
display(gradient)

# Bereken het specifiek debiet (q) gegeven een hydraulische conductiviteit (K) van 15 m/dag

Tip: $q$(m/dag) = K(m/dag) * $\Delta$h (m)/ $\Delta$x (m)

In [ ]:
hydraulic_conductivity = 15  # m/dag
specific_discharge = hydraulic_conductivity * gradient
display(specific_discharge)

# Exporteer gemiddelde grondwaterstand van `flow_mpcode_1` en `flow_mpcode_2` naar geopackage/geojson/shape file


Tip 1: Sla de gemiddelde gws op als kolom in het filters dataframe
Tip 2: filters.to_file("output.gpkg", driver="GPKG") of filters.to_file("output.json", driver="GeoJSON")

In [ ]:
filters = filters.copy()
filters["gemiddelde_gws"] = [gws_1.mean(), gws_2.mean()]
display(filters[["MpCode", "Filtnr", "gemiddelde_gws", "geometry"]])

# Lokaal exporteren kan met:
# filters.to_file("output.gpkg", driver="GPKG")
# filters.to_file("output.json", driver="GeoJSON")